# 1. Simple RNN

In [1]:
# importing the required libraries
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers 
from tensorflow.keras.datasets import imdb 
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.10.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# loading the inbuilt imdb dataset 

(x_train,y_train), (x_test,y_test) = imdb.load_data(
    num_words= 10000
)

In [4]:
# inspecting the data 

print(len(x_train))
print(len(x_test))

25000
25000


In [5]:
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


In [6]:
print(y_train[:16])
# 1 means positive review 
# 0 means negative review 

[1 0 0 1 0 0 1 0 1 0 1 0 0 0 0 0]


In [7]:
# checking the review lenghts 

print("First review length:", len(x_train[0]))
print("Second review length:", len(x_train[1]))
print("Third review length:", len(x_train[2]))

# so here everything have different lenght 
#  sequence lenght is different we cannot fed directly tensorflow wont accept this 
# so we need paddig

First review length: 218
Second review length: 189
Third review length: 141


In [8]:
# lets choose max lenght of 200 

max_length = 200 

x_train = pad_sequences(
    x_train,
    maxlen= max_length
)

x_test = pad_sequences(
    x_test,
    maxlen= max_length
)

In [9]:
x_train.shape,'  ',x_test.shape

# there are some training len greater than 200 
# if its 200 it will remove the first values for eg len is 218 it will remove the first 18 values 
# if u wanna remove the last 18 we have to use truncating='post' 

((25000, 200), '  ', (25000, 200))

In [10]:
# now creating the model 

with tf.device('/GPU:0'):
    model = keras.Sequential([
    
        layers.Input(shape=(200,)), #our batchsize may vary 32,64.. 
        # so our input per word is 200 after padding thats constant(200) thats why (200,)
    
        layers.Embedding(
            input_dim= 10000,
            output_dim= 128
        ),
        # This Embedding layer will give each words own embedding matrix 
        # 128 1D matrix [.................]
        # input_dim Think of this as how many unique words (or tokens) you can have in your vocabulary.
        # Example: If input_dim=10000, it means you can handle word IDs from 0 to 9999.
        # output_dim gives size of vector representation for each word
    
        # shape before embedding (batch_size,200) 
        # after Embedding (batch_size,200,128)
    
        layers.SimpleRNN(64),
        # here 64 referes to the number of hidden units in the RNN
        # there is no return_sequences = True so it will return the final hidden unit 64'th hidden unit will be the output
        # now the shape will be (batch_size,200,128) --> (batch_size,64)
        # 200 tokens so it will sequentially return h1 h2 h3 ... h200 in each it have [h1,h2,...h64(this 64 o/p coz simpleRNN(64))]
        # due to return_sequences = False it will return h200 alone
        # so that output will be (batch_size,64)
    
        layers.Dense(
            1,
            activation='sigmoid'
        )
        # now this layer will return only only one value like probability
        # between 0 to 1 0.10 0.83 ... 
        
        
    ])
    
    model.compile(
        optimizer= 'adam',
        loss = 'binary_crossentropy',
        metrics = ['accuracy']
    )

In [11]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 200, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 64)                12352     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 1,292,417
Trainable params: 1,292,417
Non-trainable params: 0
_________________________________________________________________


In [12]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [13]:
history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    callbacks=[early_stopping]
)
# now the model will restore the weights
# model will have best weights 


Epoch 1/5
313/313 [==============================] - 72s 222ms/step - loss: 0.5901 - accuracy: 0.6712 - val_loss: 0.5655 - val_accuracy: 0.7098
Epoch 2/5
313/313 [==============================] - 74s 238ms/step - loss: 0.3593 - accuracy: 0.8456 - val_loss: 0.5227 - val_accuracy: 0.7522
Epoch 3/5
313/313 [==============================] - 71s 228ms/step - loss: 0.1547 - accuracy: 0.9408 - val_loss: 0.5516 - val_accuracy: 0.7982
Epoch 4/5
313/313 [==============================] - 71s 226ms/step - loss: 0.0449 - accuracy: 0.9870 - val_loss: 0.7270 - val_accuracy: 0.7856


In [14]:
test_loss,test_accuracy = model.evaluate(
    x_test,
    y_test,
    batch_size=64
)

print('Test_Loss:     ',test_loss)
print('Test_Accuracy: ',test_accuracy)

391/391 [==============================] - 12s 30ms/step - loss: 0.5047 - accuracy: 0.7600
Test_Loss:      0.5046797394752502
Test_Accuracy:  0.7599999904632568


In [18]:
import numpy as np

word_index = imdb.get_word_index()

test_texts = [
    "This movie was fantastic and I loved every minute of it.",
    "Absolutely terrible, a complete waste of time and money.",
    "A decent film, but it could have been much better."
]

encoded_texts = []

for text in test_texts:
    encoded = []
    
    for word in text.lower().split():
        
        # Remove some common punctuation
        word = word.strip(".,!?;:\"'()")
        
        # Look up word in IMDB vocabulary
        index = word_index.get(word)

        if index is not None:
            # IMDB reserves 0, 1 and 2
            encoded.append(index + 3)
        else:
            # Unknown word → <UNK> token
            encoded.append(2)

    encoded_texts.append(encoded)

max_length = 200

test_sequences = pad_sequences(
    encoded_texts,
    maxlen=max_length
)

print("Input shape:", test_sequences.shape)

predictions = model.predict(
    test_sequences,
    verbose=0
)

for text, prediction in zip(test_texts, predictions):

    probability = float(prediction[0])

    if probability >= 0.5:
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    print("\n" + "=" * 70)
    print("Review:")
    print(text)

    print(f"\nPositive Probability: {probability:.4f}")
    print(f"Negative Probability: {1 - probability:.4f}")

    print(f"Prediction: {sentiment}")


Input shape: (3, 200)

Review:
This movie was fantastic and I loved every minute of it.

Positive Probability: 0.9334
Negative Probability: 0.0666
Prediction: Positive 😊

Review:
Absolutely terrible, a complete waste of time and money.

Positive Probability: 0.0300
Negative Probability: 0.9700
Prediction: Negative 😞

Review:
A decent film, but it could have been much better.

Positive Probability: 0.6550
Negative Probability: 0.3450
Prediction: Positive 😊


# 2. Stacked RNN

In [19]:
# ============================================================
# STACKED SIMPLE RNN
# ============================================================

with tf.device('/GPU:0'):

    model_stacked = keras.Sequential([

        layers.Input(shape=(200,)),
        # Input shape:
        # (batch_size, 200)
        #
        # 200 = number of tokens after padding
        # batch_size can vary: 32, 64, 128, etc.


        layers.Embedding(
            input_dim=10000,
            output_dim=128
        ),
        # Shape:
        # (batch_size, 200)
        #        ↓
        # (batch_size, 200, 128)
        #
        # Each token gets a 128-dimensional embedding.


        layers.SimpleRNN(
            64,
            return_sequences=True
        ),
        # First RNN
        #
        # 64 = hidden units
        #
        # return_sequences=True means:
        # return h1, h2, h3, ... h200
        #
        # Shape:
        # (batch_size, 200, 128)
        #        ↓
        # (batch_size, 200, 64)


        layers.SimpleRNN(
            32
        ),
        # Second RNN
        #
        # 32 = hidden units
        #
        # return_sequences=False by default
        #
        # Therefore it returns only the final hidden state h200
        #
        # Shape:
        # (batch_size, 200, 64)
        #        ↓
        # (batch_size, 32)


        layers.Dense(
            1,
            activation='sigmoid'
        )
        # Shape:
        # (batch_size, 32)
        #        ↓
        # (batch_size, 1)
        #
        # Output = probability between 0 and 1

    ])


    model_stacked.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

In [20]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model_stacked.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/5
313/313 [==============================] - 138s 437ms/step - loss: 0.4906 - accuracy: 0.7586 - val_loss: 0.5305 - val_accuracy: 0.7424
Epoch 2/5
313/313 [==============================] - 141s 450ms/step - loss: 0.3098 - accuracy: 0.8753 - val_loss: 0.4549 - val_accuracy: 0.8372
Epoch 3/5
313/313 [==============================] - 134s 428ms/step - loss: 0.2593 - accuracy: 0.8902 - val_loss: 0.4909 - val_accuracy: 0.7922
Epoch 4/5
313/313 [==============================] - 135s 432ms/step - loss: 0.1783 - accuracy: 0.9322 - val_loss: 0.5843 - val_accuracy: 0.8222


In [21]:
test_loss, test_accuracy = model_stacked.evaluate(
    x_test,
    y_test,
    batch_size = 64
)

print('Test Loss:     ',test_loss)
print('Test Accuracy: ',test_accuracy)

391/391 [==============================] - 22s 55ms/step - loss: 0.4472 - accuracy: 0.8328
Test Loss:      0.4472476840019226
Test Accuracy:  0.8328400254249573


In [22]:
predictions = model_stacked.predict(
    test_sequences,
    verbose=0
)

for text, prediction in zip(test_texts, predictions):

    probability = float(prediction[0])

    if probability >= 0.5:
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    print("\n" + "=" * 70)
    print("Review:")
    print(text)

    print(f"\nPositive Probability: {probability:.4f}")
    print(f"Negative Probability: {1 - probability:.4f}")

    print(f"Prediction: {sentiment}")



Review:
This movie was fantastic and I loved every minute of it.

Positive Probability: 0.9764
Negative Probability: 0.0236
Prediction: Positive 😊

Review:
Absolutely terrible, a complete waste of time and money.

Positive Probability: 0.4425
Negative Probability: 0.5575
Prediction: Negative 😞

Review:
A decent film, but it could have been much better.

Positive Probability: 0.9103
Negative Probability: 0.0897
Prediction: Positive 😊


# 3.LSTM

In [23]:
# now creating the model 

with tf.device('/GPU:0'):
    model_lstm = keras.Sequential([
    
        layers.Input(shape=(200,)), #our batchsize may vary 32,64.. 
        # so our input per word is 200 after padding thats constant(200) thats why (200,)
    
        layers.Embedding(
            input_dim=10000,
            output_dim=128
        ),
        # This Embedding layer will give each words own embedding matrix 
        # 128 1D matrix [.................]
        # input_dim Think of this as how many unique words (or tokens) you can have in your vocabulary.
        # Example: If input_dim=10000, it means you can handle word IDs from 0 to 9999.
        # output_dim gives size of vector representation for each word
    
        # shape before embedding (batch_size,200) 
        # after Embedding (batch_size,200,128)
    
    
        layers.LSTM(
            64,
            return_sequences=True
        ),
        # here 64 referes to the number of hidden units in the LSTM
        # we are using return_sequences = True because we are going to stack another LSTM layer
        # so this first LSTM should return all the hidden states h1,h2,h3.....h200
        # each hidden state will have 64 values because we have 64 hidden units
        # so the output shape will be (batch_size,200,64)
        #
        # shape:
        # (batch_size,200,128) --> (batch_size,200,64)
        #
        # 200 tokens so it will sequentially process
        # h1 h2 h3 ... h200
        # and return ALL of them because return_sequences = True
    
    
        layers.LSTM(
            32
        ),
        # here 32 referes to the number of hidden units in the second LSTM
        # this LSTM receives all the hidden states from the first LSTM
        #
        # input shape:
        # (batch_size,200,64)
        #
        # since return_sequences is False by default,
        # it will return only the final hidden state h200
        #
        # this final hidden state will contain 32 values
        #
        # so the shape will be:
        # (batch_size,200,64) --> (batch_size,32)
    
    
        layers.Dense(
            1,
            activation='sigmoid'
        )
        # now this layer will return only one value like probability
        # between 0 to 1 0.10 0.83 ...
        #
        # shape:
        # (batch_size,32) --> (batch_size,1)
        
        
    ])
    
    model_lstm.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

In [24]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model_lstm.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/5
313/313 [==============================] - 19s 38ms/step - loss: 0.4232 - accuracy: 0.7983 - val_loss: 0.3180 - val_accuracy: 0.8676
Epoch 2/5
313/313 [==============================] - 11s 36ms/step - loss: 0.2414 - accuracy: 0.9076 - val_loss: 0.3329 - val_accuracy: 0.8688
Epoch 3/5
313/313 [==============================] - 12s 39ms/step - loss: 0.1725 - accuracy: 0.9370 - val_loss: 0.3719 - val_accuracy: 0.8688


In [25]:
test_loss, test_accuracy = model_lstm.evaluate(
    x_test,
    y_test,
    batch_size = 64
)

print('Test Loss:     ',test_loss)
print('Test Accuracy: ',test_accuracy)

391/391 [==============================] - 7s 17ms/step - loss: 0.3226 - accuracy: 0.8656
Test Loss:      0.32259321212768555
Test Accuracy:  0.8656399846076965


In [26]:
predictions = model_lstm.predict(
    test_sequences,
    verbose=0
)

for text, prediction in zip(test_texts, predictions):

    probability = float(prediction[0])

    if probability >= 0.5:
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    print("\n" + "=" * 70)
    print("Review:")
    print(text)

    print(f"\nPositive Probability: {probability:.4f}")
    print(f"Negative Probability: {1 - probability:.4f}")

    print(f"Prediction: {sentiment}")



Review:
This movie was fantastic and I loved every minute of it.

Positive Probability: 0.9380
Negative Probability: 0.0620
Prediction: Positive 😊

Review:
Absolutely terrible, a complete waste of time and money.

Positive Probability: 0.0905
Negative Probability: 0.9095
Prediction: Negative 😞

Review:
A decent film, but it could have been much better.

Positive Probability: 0.6862
Negative Probability: 0.3138
Prediction: Positive 😊


| Metric | Simple RNN | Stacked RNN | Stacked LSTM 🏆 |
|---|---:|---:|---:|
| **Best Val Accuracy** | 79.82% | **83.72%** | **86.88%** |
| **Final Val Accuracy** | 78.56% | 82.22% | **86.88%** |
| **Test Accuracy** | 76.00% | 83.28% | **86.56%** |
| **Test Loss** | 0.5047 | 0.4472 | **0.3226** |
| **Best Val Loss** | 0.5227 | 0.4549 | **0.3180** |
| **Final Training Accuracy** | 98.70% | 93.22% | 93.70% |
| **Training Time / Epoch** | ~71 sec | ~138 sec | **~11–19 sec** |
| **Overfitting tendency** | 🔴 High | 🟠 Moderate | 🟢 Lower |
| **Overall** | Baseline | Better | 🏆 Best |